# 2. Feature Engineering

Adds engineered ratio features and encodes categorical variables.

Run this after `01_clean_data.ipynb`.

In [2]:
from pathlib import Path

def find_project_root(marker="requirements.txt", max_search_depth=4):
    """Locates the housing_project root so paths work no matter where
    Jupyter was launched from. Two strategies, tried in order:
    1. Walk UPWARD from the current directory (covers launching Jupyter
       from inside the project, e.g. from notebooks/pipeline/).
    2. Search DOWNWARD into subfolders (covers the common case of
       launching Jupyter from your home folder or Desktop, then browsing
       into the project through the Jupyter file browser -- the kernel's
       working directory stays at the launch folder, not the notebook's).
    """
    start = Path.cwd().resolve()

    # Strategy 1: search upward
    for parent in [start] + list(start.parents):
        if (parent / marker).exists():
            return parent

    # Strategy 2: search downward (breadth-first, limited depth)
    frontier = [start]
    for _ in range(max_search_depth):
        next_frontier = []
        for folder in frontier:
            try:
                subdirs = [d for d in folder.iterdir() if d.is_dir() and not d.name.startswith(".")]
            except PermissionError:
                continue
            for d in subdirs:
                if (d / marker).exists():
                    return d
                next_frontier.append(d)
        frontier = next_frontier
        if not frontier:
            break

    raise FileNotFoundError(
        f"Could not locate the project root (looking for '{marker}') starting from {start}.\n"
        f"Fix: either launch Jupyter from inside the housing_project folder, "
        f"or set PROJECT_ROOT manually below, e.g.:\n"
        f'    PROJECT_ROOT = Path(r"C:\\path\\to\\housing_project")'
    )

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

Project root: D:\Projectsinterns\housing_project


In [3]:
import pandas as pd
import numpy as np

CLEAN_PATH = PROJECT_ROOT / "data" / "housing_clean.csv"
FEATURES_OUT_PATH = PROJECT_ROOT / "data" / "housing_features.csv" 

## Load cleaned data

In [4]:
df = pd.read_csv(CLEAN_PATH)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


## Ratio features

- `rooms_per_household`
- `bedrooms_per_room`
- `population_per_household`

In [5]:
def add_ratio_features(df):
    df = df.copy()
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["population_per_household"] = df["population"] / df["households"]
    return df

## Categorical encoding

In [6]:
def encode_categorical(df, column="ocean_proximity"):
    df = df.copy()
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=False)
    df = pd.concat([df.drop(columns=[column]), dummies], axis=1)
    return df

## Log-transformed target

Useful if you want to try linear models against a less skewed target.

In [7]:
def add_log_target(df, target_col="median_house_value"):
    df = df.copy()
    df[f"log_{target_col}"] = np.log1p(df[target_col])
    return df

## Full pipeline

In [8]:
def build_features(df):
    df = add_ratio_features(df)
    df = add_log_target(df)
    df = encode_categorical(df)
    return df

df_features = build_features(df)
df_features.to_csv(FEATURES_OUT_PATH, index=False)
print(f"Saved feature-engineered dataset -> {FEATURES_OUT_PATH}  (shape: {df_features.shape})")
df_features.head()

Saved feature-engineered dataset -> D:\Projectsinterns\housing_project\data\housing_features.csv  (shape: (20640, 18))


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household,log_median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,6.984127,0.146591,2.555556,13.022766,False,False,False,True,False
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,6.238137,0.155797,2.109842,12.789687,False,False,False,True,False
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,8.288136,0.129516,2.802260,12.771673,False,False,False,True,False
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,5.817352,0.184458,2.547945,12.740520,False,False,False,True,False
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,6.281853,0.172096,2.181467,12.743154,False,False,False,True,False
